Entrenamiento y serialización de un clasificador de atrasos
Se predice si un vuelo llega con al menos 15 minutos de atraso. Se utilizan
los CSV preparados por el EDA: 8.000 registros de entrenamiento y 2.000 de prueba.
La población corresponde a vuelos no cancelados ni desviados, con atraso observado.
Los predictores están disponibles antes de la salida programada. Las definiciones
se incluyen aquí y no requieren un archivo de configuración externo.

El artefacto final incluye el preprocesamiento y Random Forest en un solo Pipeline.
Los parámetros se fijan antes de evaluar test. Esta ejecución no realiza búsqueda
de hiperparámetros ni selecciona el umbral usando resultados del conjunto de prueba.

# 1. Bibliotecas y rutas
Ejecutar todas las celdas desde la raíz del proyecto o desde notebooks/.
El notebook es autónomo. Al repetirlo se actualizan los artefactos de model/
y se conserva una copia de cada ejecución en model/versions/.
El entorno y las versiones efectivamente utilizados se registran en metadata.json.

Los reportes `balance_train.csv`, `metricas_test.csv` y `matriz_confusion.csv`
se guardan en `docs/entrenamiento/reportes/`. Las carpetas se crean automáticamente.
El modelo, los metadatos y las versiones se guardan en `model/`.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import platform
import shutil
import subprocess
import sys
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

BASE = Path.cwd()
if BASE.name.lower() == 'notebooks':
    BASE = BASE.parent
DATOS = BASE / 'data'
MODELO = BASE / 'model'
REPORTES = BASE / 'docs' / 'entrenamiento' / 'reportes'
MODELO.mkdir(exist_ok=True)
REPORTES.mkdir(parents=True, exist_ok=True)
SEMILLA = 42
MODEL_VERSION = '1.0.0'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')

# Las versiones que afectan el pipeline deben coincidir con requirements.txt.
versiones_modelo = {
    'scikit-learn': '1.8.0', 'pandas': '3.0.1', 'numpy': '2.3.5',
    'scipy': '1.16.3', 'joblib': '1.5.3', 'threadpoolctl': '3.6.0',
}
for paquete, esperada in versiones_modelo.items():
    actual = importlib.metadata.version(paquete)
    if actual != esperada:
        raise RuntimeError(f'{paquete}: se requiere {esperada}, pero está instalado {actual}. Instala requirements.txt.')
print('Python:', platform.python_version(), '| scikit-learn:', sklearn.__version__)

Python: 3.13.13 | scikit-learn: 1.8.0


# 2. Definiciones de entrada y comprobación de la separación
Las variables se enumeran en el mismo orden que en el EDA. ARRIVAL_DELAY sirve
para verificar la etiqueta, pero no se incluye en la entrada porque revelaría
la respuesta. SOURCE_ROW permite comprobar que ningún vuelo esté en ambos grupos.

Se conservan las particiones estratificadas 80/20 creadas con semilla 42 en el EDA;
no se mezclan ni se vuelven a dividir. Las comprobaciones de test validan su
integridad y no se utilizan para decidir parámetros del modelo.

In [2]:
CAT_COLS = ['AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']
NUM_COLS = ['MONTH', 'DAY', 'DAY_OF_WEEK', 'SCHEDULED_DEPARTURE_MIN',
            'SCHEDULED_TIME', 'DISTANCE']
FEATURES = CAT_COLS + NUM_COLS
TARGET = 'ARRIVAL_DELAY_15'
AUDITORIA = ['SOURCE_ROW', 'FLIGHT_DATE', 'ARRIVAL_DELAY']

train = pd.read_csv(DATOS / 'train.csv')
test = pd.read_csv(DATOS / 'test.csv')
assert len(train) == 8000 and len(test) == 2000, 'Se esperan las particiones del EDA de 10.000 vuelos.'
for nombre, tabla in [('train', train), ('test', test)]:
    assert set(FEATURES + [TARGET] + AUDITORIA).issubset(tabla.columns), nombre
    assert tabla[FEATURES + [TARGET] + AUDITORIA].notna().all().all(), nombre
    assert tabla.SOURCE_ROW.is_unique, nombre
    assert set(tabla[TARGET]) == {0, 1}, nombre
    assert tabla[TARGET].eq(tabla.ARRIVAL_DELAY.ge(15)).all(), nombre
    assert np.isfinite(tabla[NUM_COLS].to_numpy()).all(), nombre
assert set(train.SOURCE_ROW).isdisjoint(test.SOURCE_ROW)
assert not set(FEATURES).intersection(AUDITORIA)
X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
balance = y_train.value_counts().sort_index().rename_axis('clase').to_frame('vuelos')
balance['porcentaje'] = 100 * balance.vuelos / len(train)
balance.to_csv(REPORTES / 'balance_train.csv')
print('Entrenamiento:', len(train), '| Prueba:', len(test))
print(balance.round(2).to_string())

Entrenamiento: 8000 | Prueba: 2000
       vuelos  porcentaje
clase                    
0        6507       81.34
1        1493       18.66


# 3. Pipeline completo y entrenamiento
OneHotEncoder aprende las categorías solo con train y tolera categorías nuevas.
Se mantiene StandardScaler para explicitar el tratamiento de las columnas numéricas
del ejemplo del enunciado; Random Forest no necesita estandarización para funcionar.
No se realizan transformaciones aprendidas fuera del pipeline.

Se fija class_weight='balanced' para dar mayor peso a la clase menos frecuente.
Los límites de profundidad, hojas y tamaño mínimo de hoja controlan la complejidad
y ayudan a mantener pequeño el modelo. El tamaño final se verifica después de guardarlo.
Se usa el umbral predeterminado del clasificador, sin ajustarlo con test.

In [3]:
pre = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS),
    ('num', StandardScaler(), NUM_COLS),
])
clf = RandomForestClassifier(
    n_estimators=200, max_depth=12, max_leaf_nodes=256, min_samples_leaf=8,
    max_features='sqrt', class_weight='balanced', random_state=SEMILLA, n_jobs=2,
)
pipe = Pipeline([('pre', pre), ('clf', clf)])
pipe.fit(X_train, y_train)
print('Pipeline ajustado únicamente con entrenamiento.')

Pipeline ajustado únicamente con entrenamiento.


# 4. Evaluación y justificación de métricas
F1 macro promedia el F1 de ambas clases con igual peso, por lo que no oculta
completamente a los atrasos dentro de la clase mayoritaria. Recall de la clase 1
mide qué proporción de atrasos se detecta y precisión de la clase 1 mide cuántas
alertas son correctas. Ambas muestran el compromiso entre omisiones y falsas alarmas.

Average precision resume el ordenamiento por probabilidad respecto de la clase
positiva; debe interpretarse considerando su frecuencia. También se reportan
ROC AUC y exactitud. La exactitud no se utiliza sola debido al desbalance.
Un clasificador que siempre predice 0 sirve como referencia simple, sin ajustar
decisiones a partir de esta comparación. Todas las métricas siguientes son de test.

In [4]:
pred = pipe.predict(X_test)
prob = pipe.predict_proba(X_test)[:, list(pipe.classes_).index(1)]
metricas = {
    'f1_macro': float(f1_score(y_test, pred, average='macro', zero_division=0)),
    'recall_atraso': float(recall_score(y_test, pred, zero_division=0)),
    'precision_atraso': float(precision_score(y_test, pred, zero_division=0)),
    'average_precision': float(average_precision_score(y_test, prob)),
    'roc_auc': float(roc_auc_score(y_test, prob)),
    'accuracy': float(accuracy_score(y_test, pred)),
}
referencia = np.zeros(len(y_test), dtype=int)
baseline = {
    'accuracy': float(accuracy_score(y_test, referencia)),
    'f1_macro': float(f1_score(y_test, referencia, average='macro', zero_division=0)),
    'recall_atraso': 0.0,
}
pd.Series(metricas, name='valor').rename_axis('metrica').to_csv(REPORTES / 'metricas_test.csv')
matriz = confusion_matrix(y_test, pred, labels=[0, 1])
pd.DataFrame(matriz, index=['real_0', 'real_1'], columns=['pred_0', 'pred_1']).to_csv(
    REPORTES / 'matriz_confusion.csv')
print(pd.Series(metricas).round(4).to_string())
print('Referencia siempre sin atraso:', baseline)
print('Matriz de confusión: filas reales, columnas predichas; orden [0, 1]')
print(matriz)

f1_macro             0.5184
recall_atraso        0.4718
precision_atraso     0.2310
average_precision    0.2403
roc_auc              0.5876
accuracy             0.6085
Referencia siempre sin atraso: {'accuracy': 0.8135, 'f1_macro': 0.4485800937413841, 'recall_atraso': 0.0}
Matriz de confusión: filas reales, columnas predichas; orden [0, 1]
[[1041  586]
 [ 197  176]]


# 5. Serializar y verificar la inferencia en otro proceso
model.pkl contiene el Pipeline completo: ColumnTransformer y RandomForestClassifier.
Se comprueba el límite de 100 MB y se carga el archivo en un proceso Python nuevo.
Ese proceso recibe las columnas originales, sin codificar ni escalar, y debe producir
las mismas predicciones y probabilidades. También se verifica una categoría desconocida.
No se requieren clases propias ni el notebook para deserializar el artefacto.

In [5]:
ruta_modelo = MODELO / 'model.pkl'
joblib.dump(pipe, ruta_modelo, compress=3)
peso = ruta_modelo.stat().st_size
assert peso < 100_000_000, 'El modelo supera el límite de 100 MB.'
entrada_prueba = X_test.head(12).copy()
entrada_prueba.loc[entrada_prueba.index[0], 'ORIGIN_AIRPORT'] = 'IATA:ZZZ'
entrada_prueba.to_json(MODELO / 'verificacion_entrada.json', orient='records', indent=2)
esperado_pred = pipe.predict(entrada_prueba)
esperado_prob = pipe.predict_proba(entrada_prueba)
verificador = '''
from pathlib import Path
import json, joblib, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
base = Path.cwd()
modelo = joblib.load(base / 'model/model.pkl')
assert isinstance(modelo, Pipeline)
assert isinstance(modelo.named_steps['pre'], ColumnTransformer)
assert isinstance(modelo.named_steps['clf'], RandomForestClassifier)
datos = pd.read_json(base / 'model/verificacion_entrada.json')
resultado = {'pred': modelo.predict(datos).tolist(), 'prob': modelo.predict_proba(datos).tolist()}
print(json.dumps(resultado))
'''
resultado = subprocess.run([sys.executable, '-c', verificador], cwd=BASE,
                           capture_output=True, text=True, check=True)
recargado = json.loads(resultado.stdout)
np.testing.assert_array_equal(esperado_pred, recargado['pred'])
np.testing.assert_allclose(esperado_prob, recargado['prob'], rtol=1e-12, atol=1e-12)
(MODELO / 'verificacion_entrada.json').unlink()
print(f'Modelo: {peso / 1_000_000:.3f} MB. Recarga e inferencia verificadas en otro proceso.')

Modelo: 0.362 MB. Recarga e inferencia verificadas en otro proceso.


# 6. Metadatos y archivos de entorno
metadata.json conserva el orden de entrada, versiones, métricas, parámetros,
población y trazabilidad de los CSV usados. runtime.txt refleja el intérprete real.
requirements.txt está fijado y se entrega en la raíz junto a Procfile. Este último
declara el arranque de app.main:app escuchando en 0.0.0.0 y el puerto $PORT.
La implementación y pruebas HTTP de la API pertenecen a la etapa siguiente.

In [6]:
metadata = {
    'model_version': MODEL_VERSION, 'run_id': RUN_ID,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'python': platform.python_version(), 'sklearn': sklearn.__version__,
    'features': FEATURES, 'categorical_features': CAT_COLS, 'numeric_features': NUM_COLS,
    'target': TARGET, 'target_rule': 'ARRIVAL_DELAY >= 15',
    'audit_columns_excluded': AUDITORIA,
    'population': 'Vuelos no cancelados ni desviados, con atraso observado',
    'prediction_time': 'Antes de la salida programada',
    'metric': 'f1_macro', 'value': metricas['f1_macro'], 'metrics': metricas,
    'baseline_always_zero': baseline, 'confusion_matrix': matriz.tolist(),
    'class_order': pipe.classes_.tolist(), 'random_state': SEMILLA,
    'split': {'method': 'Partición estratificada 80/20 exportada por el EDA',
              'seed': SEMILLA, 'n_train': len(train), 'n_test': len(test),
              'no_shared_source_rows': True},
    'positive_rate_train': float(y_train.mean()), 'positive_rate_test': float(y_test.mean()),
    'estimator': 'RandomForestClassifier', 'parameters': clf.get_params(),
    'versions': {p: importlib.metadata.version(p) for p in versiones_modelo},
    'model_size_bytes': peso, 'full_pipeline': True, 'independent_reload_verified': True,
    'model_sha256': hashlib.sha256(ruta_modelo.read_bytes()).hexdigest(),
    'data_sha256': {name: hashlib.sha256((DATOS / name).read_bytes()).hexdigest()
                    for name in ['train.csv', 'test.csv']},
    'limitations': [
        'Datos de 2015; una separación aleatoria no demuestra desempeño en años futuros.',
        'Posible dependencia entre vuelos de rutas o fechas cercanas.',
        'Códigos IATA/BTS se mantienen separados; probabilidades no calibradas.',
        'Test ya fue utilizado en ejercicios anteriores; no es una evaluación externa nueva.',
    ],
}
(MODELO / 'metadata.json').write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')
(BASE / 'runtime.txt').write_text('python-' + platform.python_version() + '\n', encoding='utf-8')
(BASE / 'Procfile').write_text('web: uvicorn app.main:app --host 0.0.0.0 --port $PORT\n', encoding='utf-8')
print('Generados: model/model.pkl, model/metadata.json, runtime.txt y Procfile.')

# Cada ejecución conserva su modelo y entorno; los archivos de model/ son la copia actual.
archivo_version = MODELO / 'versions' / MODEL_VERSION / RUN_ID
archivo_version.mkdir(parents=True, exist_ok=False)
for origen in [ruta_modelo, MODELO / 'metadata.json', BASE / 'runtime.txt',
               BASE / 'requirements.txt', BASE / 'Procfile']:
    shutil.copyfile(origen, archivo_version / origen.name)
# Conserva el código del notebook sin resultados de ejecuciones anteriores.
fuente = BASE / 'notebooks/train.ipynb'
if fuente.exists():
    copia_notebook = json.loads(fuente.read_text(encoding='utf-8'))
    for celda in copia_notebook['cells']:
        if celda['cell_type'] == 'code':
            celda['outputs'] = []
            celda['execution_count'] = None
    (archivo_version / 'train.ipynb').write_text(
        json.dumps(copia_notebook, ensure_ascii=False, indent=1), encoding='utf-8')
print('Versión conservada en:', archivo_version.relative_to(BASE))

Generados: model/model.pkl, model/metadata.json, runtime.txt y Procfile.
Versión conservada en: model\versions\1.0.0\20260926T132639497579Z


### Versionado

`MODEL_VERSION` identifica la versión lógica del modelo y debe cambiar cuando el equipo
decida publicar una revisión. `RUN_ID` identifica cada ejecución, incluso si se repite
la misma versión. Se conservan modelo, metadatos, dependencias, runtime y Procfile en
`model/versions/<version>/<ejecucion>/`. La copia de `model/model.pkl` corresponde
a la última ejecución completada. Los hashes permiten identificar los datos y el modelo.

Git registra el código del notebook; el historial de modelos es local.
Para volver a un modelo anterior, se deben recuperar juntos el pipeline, sus metadatos
y el entorno registrado. Esta entrega no implementa despliegue o reversión automáticos.

# 7. Conclusiones
Se preserva la separación entre entrenamiento y prueba y todas las transformaciones
aprendidas se ajustan dentro del pipeline con train. Las métricas permiten valorar
tanto la detección de atrasos como las falsas alarmas, sin depender solo de exactitud.
La recarga comprueba que el artefacto serializado recibe directamente las variables
originales, incluido un aeropuerto no visto. Los resultados no garantizan desempeño
futuro: corresponden a esta partición y a la población definida.

In [7]:
print(f'F1 macro: {metricas["f1_macro"]:.4f}')
print(f'Recall de atrasos: {metricas["recall_atraso"]:.4f}')
print(f'Precisión de alertas: {metricas["precision_atraso"]:.4f}')
print(f'Archivo completo: {peso / 1_000_000:.3f} MB, inferior a 100 MB.')

F1 macro: 0.5184
Recall de atrasos: 0.4718
Precisión de alertas: 0.2310
Archivo completo: 0.362 MB, inferior a 100 MB.
